# MEGA → Google Drive Importer для Google Colab

Один Colab-ноутбук для импорта публичных MEGA-ссылок в Google Drive.

### Что умеет эта версия
- публичные `mega.nz/file/...` и `mega.nz/folder/...`;
- несколько ссылок в очереди;
- сохранение структуры папок;
- отображение квоты Google Drive;
- предварительная проверка свободного места;
- загрузка в Drive через resumable upload;
- автоматический retry ошибок до 3 раз;
- пропуск уже существующего файла при совпадении имени и размера;
- прогресс, скорость и логи;
- кнопки запуска/остановки очереди;
- **встроенная поддержка 3-х видов веб-туннелей (Colab, Cloudflare, Localtunnel)**;
- **сохранение очереди и состояния между сессиями на Google Диске**;
- **аккуратный интерфейс со скрытым кодом**;
- **навигатор по папкам Google Drive прямо в интерфейсе**.

> ✅ **Важное улучшение:** База данных очереди теперь хранится в вашем Google Диске (в папке `MegaImporter_State`). Даже если сессия Colab прервется или вы перезагрузите страницу, при следующем запуске вы не потеряете историю и статус задач.

### Запуск
1. Запустите первую ячейку установки.
2. Запустите вторую ячейку.
3. Авторизуйте Google Drive (потребуется два разрешения: для диска и для API).
4. Выберите одну из предложенных ссылок для открытия интерфейса.
5. Выберите папку сохранения через файловый менеджер, вставьте ссылки и запустите импорт.


In [1]:
#@title 🛠️ Установка зависимостей (Нажмите Play) { display-mode: "form" }
# Colab обычно использует Ubuntu. Сначала пробуем официальный пакет MEGAcmd.
# Если apt-репозиторий не содержит пакет, используем официальный .deb для текущего Ubuntu.

!apt-get update -qq
!apt-get install -y -qq curl ca-certificates python3-pip

import os, platform, subprocess, re, glob

def sh(cmd):
    print("$", " ".join(cmd))
    p = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout[-6000:])
    return p.returncode

if subprocess.run(["bash", "-lc", "command -v mega-get >/dev/null 2>&1"]).returncode != 0:
    # Определяем Ubuntu-версию.
    info = subprocess.check_output(
        ["bash", "-lc", "source /etc/os-release && echo \"$ID $VERSION_ID\""],
        text=True
    ).strip().split()
    distro, version = info[0], info[1]

    if distro == "ubuntu":
        arch = subprocess.check_output(["dpkg", "--print-architecture"], text=True).strip()
        deb_url = f"https://mega.nz/linux/repo/xUbuntu_{version}/{arch}/megacmd-xUbuntu_{version}_{arch}.deb"
        print("Пытаюсь установить:", deb_url)
        rc = sh(["bash", "-lc", f"curl -fL '{deb_url}' -o /tmp/megacmd.deb"])
        if rc == 0:
            rc = sh(["bash", "-lc", "apt-get install -y /tmp/megacmd.deb"])
        else:
            print("Официальный .deb для этой версии Ubuntu не найден.")
    else:
        print("Текущий образ не Ubuntu; автоматическая установка .deb пропущена.")

# MEGAcmd не всегда публикует пакет под каждую версию Ubuntu.
# В крайнем случае пробуем apt-пакет.
if subprocess.run(["bash", "-lc", "command -v mega-get >/dev/null 2>&1"]).returncode != 0:
    sh(["bash", "-lc", "apt-get install -y megacmd"])

# Python-зависимости
!pip -q install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib flask werkzeug psutil

# Финальная проверка.
check = subprocess.run(["bash", "-lc", "command -v mega-get && mega-get --help | head -n 8"],
                       text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
if check.returncode != 0:
    raise RuntimeError("MEGAcmd/mega-get не установлен. Проверьте логи выше.")
else:
    print("✅ Установка успешно завершена! Переходите к следующей ячейке.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package javascript-common.
(Reading database ... 118422 files and directories currently installed.)
Preparing to unpack .../0-javascript-common_11+nmu1_all.deb ...
Unpacking javascript-common (11+nmu1) ...
Selecting previously unselected package libjs-underscore.
Preparing to unpack .../1-libjs-underscore_1.13.2~dfsg-2_all.deb ...
Unpacking libjs-underscore (1.13.2~dfsg-2) ...
Selecting previously unselected package libjs-sphinxdoc.
Preparing to unpack .../2-libjs-sphinxdoc_4.3.2-1_all.deb ...
Unpacking libjs-sphinxdoc (4.3.2-1) ...
Selecting previously unselected package python3.10-dev.
Preparing to unpack .../3-python3.10-dev_3.10.12-1~22.04.16_amd64.deb ...
Unpacking python3.10-dev (3.10.12-1~22.04.16) ...
Selecting previously unselected package python3-dev.
Prepari

In [ ]:
#@title 🚀 Запуск: скачать код и запустить { display-mode: "form" }

GITHUB_REPO = "YOUR_USERNAME/mega-to-gdrive"  # <- замените на свой репозиторий
BRANCH      = "main"

import subprocess, sys, shutil

# Удаляем старую копию кода (чтобы всегда тянуть актуальную версию)
shutil.rmtree("/content/app", ignore_errors=True)

# Клонируем репозиторий
subprocess.run(
    ["git", "clone", "--depth=1", "--branch", BRANCH,
     f"https://github.com/{GITHUB_REPO}.git", "/content/app"],
    check=True,
)

# Добавляем путь к пакету и запускаем
sys.path.insert(0, "/content/app")
from mega_importer.server import run
run()
